# 01 — Trip Purpose VMT by Geographic Cell

**Source:** NHTS 2022 (tripv2pub.csv)  
**Geography:** Census Division (9) × MSA Category (4) = up to 36 cells  
**Output:** One 4-panel map per trip purpose, saved to `outputs/maps/`

Each panel shows Census Division polygons colored by weighted VMT share for that MSA category.

## 0. Imports and paths

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
from pathlib import Path

DATA_DIR   = Path(r"C:\Users\m.kroetz\OneDrive - Systematica\HONDA\repo\data")
TRIP_CSV   = DATA_DIR / "NHTS" / "csv" / "tripv2pub.csv"
STATE_SHP  = DATA_DIR / "tl_2024_us_state" / "tl_2024_us_state.shp"
OUTPUT_DIR = Path(r"C:\Users\m.kroetz\OneDrive - Systematica\HONDA\repo\outputs\maps")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load trip data

In [ ]:
trips = pd.read_csv(TRIP_CSV, low_memory=False)

# Keep only the columns we need
cols = ["HOUSEID", "PERSONID", "WHYTO", "TRPMILES", "WTTRDFIN",
        "CENSUS_D", "MSACAT", "TRPTRANS", "DRVR_FLG"]
trips = trips[[c for c in cols if c in trips.columns]].copy()

# Drop missing values on key fields
trips = trips.dropna(subset=["WHYTO", "TRPMILES", "WTTRDFIN", "CENSUS_D", "MSACAT"])

# Remove invalid codes
trips = trips[(trips["WHYTO"] > 0) & (trips["TRPMILES"] > 0) & (trips["WTTRDFIN"] > 0)]
trips = trips[(trips["CENSUS_D"] > 0) & (trips["MSACAT"] > 0)]

trips["WHYTO"]    = trips["WHYTO"].astype(int)
trips["CENSUS_D"] = trips["CENSUS_D"].astype(int)
trips["MSACAT"]   = trips["MSACAT"].astype(int)

print(f"Trips loaded: {len(trips):,}")
trips.head(3)

## 2. Label trip purposes and MSA categories

In [ ]:
PURPOSE_LABELS = {
    2:  "Work from home",
    3:  "Commute (work)",
    4:  "Work — drop-off/pick-up",
    5:  "Other work-related",
    6:  "School",
    7:  "Child / adult care",
    8:  "Volunteer",
    9:  "Change transport",
    10: "Drop off / pick up",
    11: "Healthcare",
    12: "Buy meals",
    13: "Shopping",
    14: "Other errands",
    15: "Recreation",
    16: "Exercise",
    17: "Visit friends / relatives",
    18: "Religious / community",
    19: "Rest / vacation",
    97: "Other",
}

MSA_LABELS = {
    1: "Not in MSA",
    2: "MSA < 1M",
    3: "MSA ≥ 1M, no heavy rail",
    4: "MSA ≥ 1M, with heavy rail",
}

DIVISION_NAMES = {
    1: "New England",
    2: "Middle Atlantic",
    3: "East North Central",
    4: "West North Central",
    5: "South Atlantic",
    6: "East South Central",
    7: "West South Central",
    8: "Mountain",
    9: "Pacific",
}

trips["purpose_label"] = trips["WHYTO"].map(PURPOSE_LABELS)
trips["msa_label"]     = trips["MSACAT"].map(MSA_LABELS)

# Weighted VMT per trip = miles × trip weight
trips["wt_vmt"] = trips["TRPMILES"] * trips["WTTRDFIN"]

## 3. Aggregate VMT by division × MSA category × trip purpose

In [ ]:
agg = (
    trips
    .groupby(["CENSUS_D", "MSACAT", "WHYTO", "purpose_label", "msa_label"])["wt_vmt"]
    .sum()
    .reset_index()
    .rename(columns={"wt_vmt": "total_wt_vmt"})
)

print(f"Aggregation rows: {len(agg)}")
agg.head()

## 4. Load state shapefile and dissolve to Census Divisions

In [ ]:
states = gpd.read_file(STATE_SHP)

# Keep continental US only (exclude AK=02, HI=15, territories)
EXCLUDE_FIPS = {"02", "15", "60", "66", "69", "72", "78"}
states = states[~states["STATEFP"].isin(EXCLUDE_FIPS)].copy()

states["DIVISION"] = states["DIVISION"].astype(int)

divisions = states.dissolve(by="DIVISION", as_index=False)[["DIVISION", "geometry"]]
divisions = divisions.to_crs(epsg=5070)  # Albers Equal Area CONUS

print("Divisions:", sorted(divisions["DIVISION"].tolist()))

## 5. Map function

In [ ]:
def plot_purpose_map(purpose_code, purpose_name, agg_df, divisions_gdf, output_dir):
    """4-panel map (one per MSA category) showing weighted VMT by Census Division."""

    subset = agg_df[agg_df["WHYTO"] == purpose_code].copy()
    if subset.empty:
        print(f"  No data for purpose {purpose_code} — skipping")
        return

    msa_cats = [1, 2, 3, 4]
    msa_names = [
        "Not in MSA",
        "MSA < 1M",
        "MSA ≥ 1M\nno heavy rail",
        "MSA ≥ 1M\nwith heavy rail",
    ]

    # Global max VMT across all MSA cats for this purpose (shared colorscale)
    global_max = subset["total_wt_vmt"].max()

    fig, axes = plt.subplots(1, 4, figsize=(22, 5))
    fig.patch.set_facecolor("#f8f8f6")

    cmap = plt.cm.YlOrRd
    norm = mcolors.Normalize(vmin=0, vmax=global_max)

    for ax, msa_code, msa_name in zip(axes, msa_cats, msa_names):
        panel_data = subset[subset["MSACAT"] == msa_code][["CENSUS_D", "total_wt_vmt"]]
        gdf = divisions_gdf.merge(panel_data, left_on="DIVISION", right_on="CENSUS_D", how="left")
        gdf["total_wt_vmt"] = gdf["total_wt_vmt"].fillna(0)

        gdf.plot(
            column="total_wt_vmt",
            ax=ax,
            cmap=cmap,
            norm=norm,
            edgecolor="white",
            linewidth=0.6,
            missing_kwds={"color": "#e0e0e0"},
        )

        ax.set_title(msa_name, fontsize=10, fontweight="bold", pad=6)
        ax.axis("off")
        ax.set_facecolor("#f8f8f6")

    # Shared colorbar
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=axes, fraction=0.015, pad=0.02)
    cbar.set_label("Weighted VMT (person-miles)", fontsize=9)
    cbar.ax.yaxis.set_major_formatter(
        plt.FuncFormatter(lambda x, _: f"{x/1e9:.1f}B" if x >= 1e9 else f"{x/1e6:.0f}M")
    )

    fig.suptitle(
        f"Weighted VMT by Census Division — {purpose_name}",
        fontsize=13, fontweight="bold", y=1.01
    )
    fig.text(
        0.5, -0.02,
        "Source: NHTS 2022 | Geography: Census Division × MSA Category",
        ha="center", fontsize=8, color="#666666"
    )

    fname = f"{str(purpose_code).zfill(2)}_{purpose_name.lower().replace(' ', '_').replace('/', '-')}.png"
    fpath = output_dir / fname
    fig.savefig(fpath, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(fig)
    print(f"  Saved: {fpath.name}")

## 6. Generate all maps

In [ ]:
print("Generating maps...")
for code, label in PURPOSE_LABELS.items():
    print(f"  Purpose {code}: {label}")
    plot_purpose_map(code, label, agg, divisions, OUTPUT_DIR)

print(f"\nDone. Maps saved to: {OUTPUT_DIR}")

## 7. Summary table — total weighted VMT by purpose (national)

In [ ]:
summary = (
    agg
    .groupby(["WHYTO", "purpose_label"])["total_wt_vmt"]
    .sum()
    .reset_index()
    .sort_values("total_wt_vmt", ascending=False)
)
summary["vmt_share_pct"] = (summary["total_wt_vmt"] / summary["total_wt_vmt"].sum() * 100).round(1)
summary["total_wt_vmt_B"] = (summary["total_wt_vmt"] / 1e9).round(2)

summary[["WHYTO", "purpose_label", "total_wt_vmt_B", "vmt_share_pct"]].rename(columns={
    "WHYTO": "Code",
    "purpose_label": "Trip Purpose",
    "total_wt_vmt_B": "Weighted VMT (B miles)",
    "vmt_share_pct": "Share (%)"
})